# 📊 Instagram Etkileşim Saatini Analiz Eden Uygulama

Bu proje, belirtilen bir Instagram hesabının gönderilerini çekerek **hangi gün**, **hangi saat** ve **hangi ayda** paylaşım yapıldığını analiz eder. Elde edilen veriler bir **Excel dosyasına** yazılır ve yüksek beğeni alan gönderiler koşullu biçimlendirme ile **yeşil renkte** vurgulanır.

Aşağıdaki hücreleri **sırasıyla (Shift + Enter)** çalıştırarak uygulamanın çalışma mantığını adım adım inceleyebilirsiniz.

## 1. Adım: Gerekli Kütüphanelerin İçe Aktarılması

Bu projede kullandığımız kütüphaneler ve görevleri:

| Kütüphane | Görevi |
|-----------|--------|
| `instaloader` | Instagram profilinden gönderi verilerini çekmek |
| `pandas` | Verileri tablo yapısına (DataFrame) dönüştürüp Excel'e yazmak |
| `openpyxl` | Excel dosyasını açıp koşullu biçimlendirme uygulamak |
| `getpass` | Şifreyi terminalde gizli olarak almak |
| `locale` | Tarih bilgilerini Türkçe göstermek (Pazartesi, Ocak vb.) |
| `os` | Dosya yolu işlemleri |

In [ ]:
import os
import instaloader
import locale
import pandas as pd
from datetime import datetime
import getpass

## 2. Adım: Türkçe Yerel Ayar (Locale) Yapılandırması

Python'un `strftime()` fonksiyonu tarih formatlarken gün ve ay isimlerini **işletim sisteminin dil ayarına** göre yazar. Varsayılan olarak İngilizce gelir (Monday, January vb.).

Bunu Türkçe yapmak için `locale.setlocale()` ile Türkçe yerel ayar aktif edilir:
- **Linux/Mac:** `tr_TR.UTF-8`
- **Windows:** `Turkish_Türkiye.1254`

In [ ]:
# Türkçe yerel ayar yapılandırması
try:
    locale.setlocale(locale.LC_ALL, "tr_TR.UTF-8")
    print("Locale ayarlandı: tr_TR.UTF-8")
except locale.Error:
    try:
        locale.setlocale(locale.LC_ALL, "Turkish_Türkiye.1254")
        print("Locale ayarlandı: Turkish_Türkiye.1254")
    except locale.Error:
        print("Uyarı: Türkçe yerel ayar bulunamadı.")

# Test: Bugünün tarihini Türkçe formatlayalım
print(f"Bugün: {datetime.now().strftime('%A, %d %B %Y')}")

## 3. Adım: Instagram Oturumu Açma (Login)

Instagram, anonim (giriş yapmadan) yapılan API isteklerini engellemektedir. Bu yüzden verileri çekebilmek için önce kendi Instagram hesabımızla oturum açmamız gerekir.

**Kullanılan Teknikler:**
- `getpass.getpass()`: Şifreyi terminalde gizli tutar (yıldız gösterir, ekranda şifre görünmez)
- `L.login()`: Instaloader kütüphanesinin oturum açma fonksiyonu
- `try-except` blokları: Yanlış şifre, 2FA gibi hataları yakalar

In [ ]:
# Instaloader örneği oluştur
L = instaloader.Instaloader()

# Giriş bilgilerini al
print("\n--- Instagram Giriş ---")
ig_username = input("Instagram kullanıcı adınız: ")
ig_password = getpass.getpass("Instagram şifreniz (gizli): ")

# Oturum açma denemesi
try:
    L.login(ig_username, ig_password)
    print("Giriş başarılı!")
except instaloader.exceptions.BadCredentialsException:
    print("Hata: Kullanıcı adı veya şifre yanlış!")
except instaloader.exceptions.TwoFactorAuthRequiredException:
    print("Hata: İki faktörlü doğrulama gerekli.")
except Exception as e:
    print(f"Giriş hatası: {e}")

## 4. Adım: Profil Yükleme ve Gönderi Verilerini Toplama

Oturum açıldıktan sonra, analiz edilecek Instagram kullanıcı adı ve gönderi sayısı alınır. Ardından `Profile.from_username()` ile profil yüklenir ve `get_posts()` ile gönderilere erişilir.

Her gönderiden şu veriler çekilir:
- **Gün**: Gönderinin paylaşıldığı gün adı (`strftime("%A")` → Pazartesi, Salı...)
- **Ay**: Gönderinin paylaşıldığı ay adı (`strftime("%B")` → Ocak, Şubat...)
- **Yıl**: Gönderinin paylaşıldığı yıl (`strftime("%Y")` → 2024, 2025...)
- **Beğeni Sayısı**: Toplam beğeni adedi (`post.likes`)
- **Saat**: Paylaşım saati (`strftime("%H:%M:%S")` → 14:30:00)
- **Gönderi Linki**: Doğrudan gönderi URL'si (`post.shortcode` kullanılarak)

In [ ]:
# Analiz edilecek hesap bilgilerini al
username = input("Analiz edilecek Instagram kullanıcı adı: ")
count = int(input("Analiz edilecek gönderi sayısı: "))

# Profili yükle
try:
    profile = instaloader.Profile.from_username(L.context, username)
    posts = profile.get_posts()
    print(f"Profil yüklendi: {profile.full_name} (@{username})")
    print(f"Toplam gönderi: {profile.mediacount}, Takipçi: {profile.followers}")
except instaloader.exceptions.ProfileNotExistsException:
    print(f"Hata: '{username}' adlı profil bulunamadı!")
except Exception as e:
    print(f"Profil yükleme hatası: {e}")

In [ ]:
# Gönderi verilerini toplama döngüsü
post_data = []

for i, post in enumerate(posts):
    if i >= count:
        break

    post_info = {
        "Gün": post.date.strftime("%A"),
        "Ay": post.date.strftime("%B"),
        "Yıl": post.date.strftime("%Y"),
        "Beğeni Sayısı": post.likes,
        "Saat": post.date.strftime("%H:%M:%S"),
        "Gönderi Linki": f'https://instagram.com/p/{post.shortcode}'
    }
    post_data.append(post_info)
    print(f"Gönderi {i + 1}/{count} tamamlandı...")

print(f"\nToplam {len(post_data)} gönderi verisi toplandı.")

## 5. Adım: Pandas DataFrame Oluşturma ve Türkçe Karakter Düzeltmesi

Toplanan veriler bir Pandas `DataFrame`'e dönüştürülür. DataFrame, verileri tablo yapısında (satır × sütun) tutarak analiz ve dışa aktarma işlemlerini kolaylaştırır.

Türkçe karakterlerin (ç, ş, ğ, ü, ö, ı) Excel'de bozulmadan görünmesi için `.encode('utf-8').decode('utf-8')` dönüşümü uygulanır.

In [ ]:
# DataFrame oluştur
df = pd.DataFrame(post_data)

# Türkçe karakter düzeltmesi
df['Gün'] = df['Gün'].str.encode('utf-8').str.decode('utf-8')
df['Ay'] = df['Ay'].str.encode('utf-8').str.decode('utf-8')

# İlk 5 satırı önizleme olarak göster
print("\nÖnizleme (İlk 5 gönderi):")
df.head()

## 6. Adım: Excel Dosyasına Yazma ve Koşullu Biçimlendirme

Veriler önce `pandas.to_excel()` ile Excel dosyasına yazılır. Ardından `openpyxl` kütüphanesi ile dosya tekrar açılıp **koşullu biçimlendirme** uygulanır:

- Beğeni sayısı **2000 ve üstü** olan hücreler **yeşil** renkle vurgulanır
- `PatternFill` nesnesi ile hücre arka plan rengi değiştirilir
- `iter_rows()` ile satır satır dolaşılarak koşul kontrol edilir

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

# Excel dosya yolunu belirle (script klasörüne kaydet)
script_dir = os.path.dirname(os.path.abspath("__file__"))
excel_file = os.path.join(script_dir, 'instagram_analist.xlsx')

# DataFrame'i Excel'e yaz
df.to_excel(excel_file, index=False, engine='openpyxl')
print(f"Veriler Excel dosyasına yazıldı: {excel_file}")

# Excel dosyasını openpyxl ile tekrar aç
wb = load_workbook(excel_file)
ws = wb.active

# Yeşil renk dolgu tanımı
yesil_dolgu = PatternFill(start_color="00FF00", end_color="00FF00", fill_type="solid")

# Beğeni Sayısı sütunu (4. sütun) üzerinde koşullu biçimlendirme
for row in ws.iter_rows(min_row=2, min_col=4, max_row=len(df) + 1, max_col=4):
    for cell in row:
        if cell.value >= 2000:
            cell.fill = yesil_dolgu

# Biçimlendirilmiş dosyayı kaydet
wb.save(excel_file)
print("Koşullu biçimlendirme uygulandı: Beğeni >= 2000 → Yeşil")
print(f"\nSonuç dosyası: {excel_file}")

---

# 🔄 Programın Çalışma Akışı ve Yapısı

Uygulama çalıştırıldığında şu adımları sırasıyla takip eder:

1. **Locale Ayarı**: İşletim sistemine göre Türkçe yerel ayar etkinleştirilir. `strftime()` ile üretilen gün ve ay isimleri Türkçe görünür.

2. **Instagram Girişi**: Kullanıcıdan kendi Instagram hesap bilgileri alınır. `getpass` ile şifre gizli tutulur. `L.login()` ile Instagram oturumu açılır.

3. **Profil Yükleme**: Analiz edilecek hedef kullanıcı adı alınır. `Profile.from_username()` ile profil yüklenir ve `get_posts()` ile gönderi iteratörü oluşturulur.

4. **Veri Toplama**: `enumerate()` döngüsü ile belirtilen sayıda gönderi taranır. Her gönderiden gün, ay, yıl, saat, beğeni sayısı ve gönderi linki çekilir.

5. **DataFrame Oluşturma**: Toplanan sözlük listesi Pandas `DataFrame`'e dönüştürülür. Türkçe karakterler UTF-8 encode/decode ile düzeltilir.

6. **Excel'e Yazma**: `to_excel()` ile veriler `.xlsx` formatında kaydedilir. `openpyxl` motoru kullanılır.

7. **Koşullu Biçimlendirme**: Excel dosyası `openpyxl.load_workbook()` ile tekrar açılır. Beğeni sayısı 2000 ve üstü olan hücreler `PatternFill` ile yeşil renkte boyanır.

8. **Sonuç**: Kullanıcıya işlem tamamlandı bilgisi verilir ve dosya yolu gösterilir.

---

## 📌 Önemli Notlar

- Instagram'ın API limitleri nedeniyle çok fazla gönderi çekilmeye çalışıldığında geçici engelleme uygulanabilir.
- Gizli (private) profillerin verileri yalnızca takipçileri tarafından çekilebilir.
- İki faktörlü doğrulama (2FA) aktifse, Instagram uygulamasından gelen onay kodunu girmek gerekir.
- `strftime()` format kodları: `%A` = gün adı, `%B` = ay adı, `%Y` = yıl, `%H:%M:%S` = saat.